In [12]:
import pandas as pd
import numpy as np
import seaborn as sns

data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
!wget $data

df = pd.read_csv(data)
df.info()

--2026-10-12 20:56:52--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.2’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.002s  

2026-10-12 20:56:52 (120 MB/s) - ‘course_lead_scoring_2026.csv.2’ saved [278746/278746]

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employm

In [13]:
# Data Prep
categorical = df.select_dtypes(include='string').columns.tolist()
numerical = df.select_dtypes(include='number').columns.tolist()

df.isna().sum()

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

In [14]:
# Q1
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [15]:
# Q2
corr_matrix = df[numerical].corr()
corr_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [16]:
# split
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_test['converted']
del df_val['converted']

In [17]:
# Q3
from sklearn.metrics import mutual_info_score

for col in categorical:
    score = mutual_info_score(df_train[col], y_train)
    print(col, round(score, 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


In [45]:
# Q4
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
# numerical.remove('converted')

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# print(model.intercept_[0],model.coef_[0].round(3))
y_pred = model.predict_proba(X_val)[:, 1]

converted_prob = (y_pred >= 0.5)

df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = converted_prob.astype(int)
df_pred['actual'] = y_val

df_pred['correct'] = df_pred.prediction == df_pred.actual
print(df_pred.correct.mean())
accuracy = (y_val == converted_prob).mean()

print(round(float(accuracy), 2))

0.645
0.65


In [46]:
# Q5
wo_lead_source = [x for x in categorical if x != 'lead_source']
wo_number_of_courses_viewed = [x for x in numerical if x != 'number_of_courses_viewed']
wo_interaction_count = [x for x in numerical if x != 'interaction_count']

dv_wo_lead_source = DictVectorizer(sparse=False)
dv_wo_number_of_courses_viewed = DictVectorizer(sparse=False)
dv_wo_interaction_count = DictVectorizer(sparse=False)

train_dict_wo_lead_source = df_train[wo_lead_source + numerical].to_dict(orient='records')
X_train_wo_lead_source = dv_wo_lead_source.fit_transform(train_dict_wo_lead_source)
train_dict_wo_number_of_courses_viewed = df_train[categorical + wo_number_of_courses_viewed].to_dict(orient='records')
X_train_wo_number_of_courses_viewed = dv_wo_number_of_courses_viewed.fit_transform(train_dict_wo_number_of_courses_viewed)
train_dict_wo_interaction_count = df_train[categorical + wo_interaction_count].to_dict(orient='records')
X_train_wo_interaction_count = dv_wo_interaction_count.fit_transform(train_dict_wo_interaction_count)

val_dict_wo_lead_source = df_val[wo_lead_source + numerical].to_dict(orient='records')
X_val_wo_lead_source = dv_wo_lead_source.transform(val_dict_wo_lead_source)
val_dict_wo_number_of_courses_viewed = df_val[categorical + wo_number_of_courses_viewed].to_dict(orient='records')
X_val_wo_number_of_courses_viewed = dv_wo_number_of_courses_viewed.transform(val_dict_wo_number_of_courses_viewed)
val_dict_wo_interaction_count = df_val[categorical + wo_interaction_count].to_dict(orient='records')
X_val_wo_interaction_count = dv_wo_interaction_count.transform(val_dict_wo_interaction_count)

model_wo_lead_source = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_wo_lead_source.fit(X_train_wo_lead_source, y_train)
model_wo_number_of_courses_viewed = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_wo_number_of_courses_viewed.fit(X_train_wo_number_of_courses_viewed, y_train)
model_wo_interaction_count = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_wo_interaction_count.fit(X_train_wo_interaction_count, y_train)

y_pred_wo_lead_source = model_wo_lead_source.predict_proba(X_val_wo_lead_source)[:, 1]
y_pred_wo_number_of_courses_viewed = model_wo_number_of_courses_viewed.predict_proba(X_val_wo_number_of_courses_viewed)[:, 1]
y_pred_wo_interaction_count = model_wo_interaction_count.predict_proba(X_val_wo_interaction_count)[:, 1]

converted_prob_wo_lead_source = (y_pred_wo_lead_source >= 0.5)
converted_prob_wo_number_of_courses_viewed = (y_pred_wo_number_of_courses_viewed >= 0.5)
converted_prob_wo_interaction_count = (y_pred_wo_interaction_count >= 0.5)

accuracy_wo_lead_source = (y_val == converted_prob_wo_lead_source).mean()
accuracy_wo_number_of_courses_viewed = (y_val == converted_prob_wo_number_of_courses_viewed).mean()
accuracy_wo_interaction_count = (y_val == converted_prob_wo_interaction_count).mean()

print('wo_lead_source: ',round(accuracy - accuracy_wo_lead_source,3))
print('wo_number_of_courses_viewed: ',round(accuracy - accuracy_wo_number_of_courses_viewed,3))
print('wo_interaction_count: ',round(accuracy - accuracy_wo_interaction_count,3))

wo_lead_source:  0.003
wo_number_of_courses_viewed:  0.002
wo_interaction_count:  0.044


In [53]:
# Q6
C_params = [0.000001, 0.00001, 0.0001, 0.001]

def train_model(C,ac):
    dv = DictVectorizer(sparse=False)

    train_dict = df_train[categorical + numerical].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    
    val_dict = df_val[categorical + numerical].to_dict(orient='records')
    X_val = dv.transform(val_dict)
    
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    
    converted_prob = (y_pred >= 0.5)
    
    df_pred = pd.DataFrame()
    df_pred['probability'] = y_pred
    df_pred['prediction'] = converted_prob.astype(int)
    df_pred['actual'] = y_val
    
    df_pred['correct'] = df_pred.prediction == df_pred.actual
    accuracy1 = (y_val == converted_prob).mean()
    
    print(f'{C:.6f}: ', round(float(ac - accuracy1), 3))

for C in C_params:
    train_model(C,accuracy)

0.000001:  0.047
0.000010:  0.047
0.000100:  0.032
0.001000:  0.0
